# OpenAI Exercise: Multi-Model SDR with Guardrails

This notebook is based on concepts taught in `2_openai` and completes the exercise from the second-last cell of `3_lab3.ipynb`:
- try different models,
- add more input and output guardrails,
- use structured outputs for email generation.


## Concept Consolidation From `2_openai`

1. `1_lab1`: OpenAI Agents SDK basics (`Agent`, `Runner`, `trace`) with a simple single-agent run.
2. `2_lab2`: Agentic workflow for SDR: parallel candidate generation, tool usage (`@function_tool`), manager orchestration, and handoffs.
3. `3_lab3`: model abstraction (`OpenAIChatCompletionsModel`), structured outputs with Pydantic, and guardrails.
4. `4_lab4`: deep research pattern with planner/search/writer specialization and reusable async workflow functions.

Applied in this notebook:
- Three OpenAI models for different writing styles.
- Structured email outputs via Pydantic schema.
- Two input guardrails and two output guardrails.


In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    set_default_openai_client,
    set_default_openai_api,
    OpenAIChatCompletionsModel,
    input_guardrail,
    output_guardrail,
    GuardrailFunctionOutput,
)
from agents.exceptions import InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered
import os


In [ ]:
# Environment setup (using OpenAI key as requested)

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing from .env")

openai_client = AsyncOpenAI(api_key=openai_api_key)
set_default_openai_client(openai_client, use_for_tracing=True)
set_default_openai_api("chat_completions")


In [ ]:
# Different models (exercise requirement: try different models)

MODEL_PROFESSIONAL = OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=openai_client)
MODEL_CREATIVE = OpenAIChatCompletionsModel(model="gpt-4.1-mini", openai_client=openai_client)
MODEL_CONCISE = OpenAIChatCompletionsModel(model="gpt-4.1-nano", openai_client=openai_client)
MODEL_GUARDRAIL = "gpt-4o-mini"


In [ ]:
# Structured outputs (exercise requirement: structured email generation)

class ColdEmail(BaseModel):
    subject: str = Field(description="Compelling subject line under 60 chars")
    body_markdown: str = Field(description="Email body in markdown")
    tone: str = Field(description="Tone used in the email")
    cta: str = Field(description="Clear call to action")


class NameCheckOutput(BaseModel):
    has_personal_name: bool
    reason: str


class PromptSafetyOutput(BaseModel):
    unsafe_or_irrelevant: bool
    reason: str


class QualityCheckOutput(BaseModel):
    unacceptable: bool
    reason: str


class ComplianceCheckOutput(BaseModel):
    risky_claims: bool
    reason: str


In [ ]:
# Sales agents using different models

instructions_professional = """
You are a professional SDR for ComplAI.
Generate a serious, credible cold sales email.
Output MUST follow the ColdEmail schema.
"""

instructions_creative = """
You are a creative SDR for ComplAI.
Generate an engaging and memorable cold sales email.
Output MUST follow the ColdEmail schema.
"""

instructions_concise = """
You are a concise SDR for ComplAI.
Generate a short, direct cold sales email.
Output MUST follow the ColdEmail schema.
"""

sales_agent_1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions_professional,
    model=MODEL_PROFESSIONAL,
    output_type=ColdEmail,
)

sales_agent_2 = Agent(
    name="Creative Sales Agent",
    instructions=instructions_creative,
    model=MODEL_CREATIVE,
    output_type=ColdEmail,
)

sales_agent_3 = Agent(
    name="Concise Sales Agent",
    instructions=instructions_concise,
    model=MODEL_CONCISE,
    output_type=ColdEmail,
)


In [ ]:
# Tools: convert agents to tools + preview sender tool

description = "Generate one cold sales email draft in the ColdEmail schema"
tool1 = sales_agent_1.as_tool(tool_name="sales_agent_1", tool_description=description)
tool2 = sales_agent_2.as_tool(tool_name="sales_agent_2", tool_description=description)
tool3 = sales_agent_3.as_tool(tool_name="sales_agent_3", tool_description=description)


@function_tool
def send_preview_email(subject: str, body_markdown: str) -> dict:
    """Preview-send an email (no external provider call)."""
    print("\n=== PREVIEW EMAIL ===")
    print("Subject:", subject)
    print("Body:\n", body_markdown)
    print("=== END PREVIEW ===\n")
    return {"status": "preview_sent"}


In [ ]:
# Guardrail agents

name_guardrail_agent = Agent(
    name="Name Guardrail",
    instructions="Detect whether the message includes a personal name to target.",
    output_type=NameCheckOutput,
    model=MODEL_GUARDRAIL,
)

prompt_guardrail_agent = Agent(
    name="Prompt Safety Guardrail",
    instructions=(
        "Mark unsafe_or_irrelevant as true if the request asks for unethical content, "
        "spammy deception, or is not about writing a cold sales email."
    ),
    output_type=PromptSafetyOutput,
    model=MODEL_GUARDRAIL,
)

quality_guardrail_agent = Agent(
    name="Output Quality Guardrail",
    instructions=(
        "Given a generated email, mark unacceptable true if it lacks a clear CTA, "
        "is unprofessional, or too vague to send."
    ),
    output_type=QualityCheckOutput,
    model=MODEL_GUARDRAIL,
)

compliance_guardrail_agent = Agent(
    name="Compliance Guardrail",
    instructions=(
        "Given a generated email, mark risky_claims true if it contains fabricated guarantees, "
        "false compliance promises, or deceptive claims."
    ),
    output_type=ComplianceCheckOutput,
    model=MODEL_GUARDRAIL,
)


In [ ]:
# Input guardrails (exercise requirement: more input guardrails)

@input_guardrail
async def guardrail_against_personal_names(ctx, agent, message):
    result = await Runner.run(name_guardrail_agent, message, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason},
        tripwire_triggered=check.has_personal_name,
    )


@input_guardrail
async def guardrail_prompt_safety(ctx, agent, message):
    result = await Runner.run(prompt_guardrail_agent, message, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason},
        tripwire_triggered=check.unsafe_or_irrelevant,
    )


In [ ]:
# Output guardrails (exercise requirement: add output guardrails)

@output_guardrail
async def guardrail_output_quality(ctx, agent, output):
    result = await Runner.run(quality_guardrail_agent, str(output), context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason},
        tripwire_triggered=check.unacceptable,
    )


@output_guardrail
async def guardrail_output_compliance(ctx, agent, output):
    result = await Runner.run(compliance_guardrail_agent, str(output), context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info={"reason": check.reason},
        tripwire_triggered=check.risky_claims,
    )


In [ ]:
# Manager agent

manager_instructions = """
You are Sales Manager at ComplAI.
1. Use all three sales_agent tools to generate draft candidates.
2. Select the single best candidate for response likelihood and professionalism.
3. Use send_preview_email with the final subject and body_markdown.
4. Return the selected draft in the ColdEmail schema.
Do not target a person by name.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=[tool1, tool2, tool3, send_preview_email],
    model="gpt-4o-mini",
    output_type=ColdEmail,
    input_guardrails=[guardrail_against_personal_names, guardrail_prompt_safety],
    output_guardrails=[guardrail_output_quality, guardrail_output_compliance],
)


In [ ]:
# Run example

message = "Write a cold sales email for a CTO at a mid-size fintech. Sender is Head of Business Development."

try:
    with trace("Week2 Exercise - Multi-Model SDR with Guardrails"):
        result = await Runner.run(sales_manager, message)

    final_email = result.final_output
    print(final_email)

except InputGuardrailTripwireTriggered as e:
    print("Input guardrail triggered:", e)

except OutputGuardrailTripwireTriggered as e:
    print("Output guardrail triggered:", e)


## Notes

- If an input guardrail triggers, adjust the prompt (for example, remove personal names).
- If an output guardrail triggers, rerun with tighter instructions.
- This implementation stays within the tools and concepts taught in the `2_openai` labs.
